# 경마 예측 모델 v1 — 해부와 개선 실험

이 노트북은 서비스의 **프로덕션 예측 코드**(`lib/predict/model-v1.ts`)를 그대로 import해서
실제 KRA 데이터로 실행·백테스트·튜닝하는 연구 환경입니다.

> **파이썬 개발자를 위한 안내** — 이 노트북은 TypeScript(Deno 커널)로 작성됐지만,
> 파이썬과 거의 1:1로 대응됩니다:
>
> | Python | TypeScript(Deno) |
> |---|---|
> | `import x from y` | `import { x } from "./y.ts"` (확장자 필수) |
> | `x = 1` | `const x = 1` (재할당 시 `let`) |
> | `lambda h: h.prob` | `(h) => h.prob` |
> | `[f(x) for x in xs]` | `xs.map((x) => f(x))` |
> | `[x for x in xs if c(x)]` | `xs.filter((x) => c(x))` |
> | `sum(xs)` | `xs.reduce((a, b) => a + b, 0)` |
> | `f-string f"{x}"` | 템플릿 리터럴 `` `${x}` `` |
> | `dict.get(k)` | `map.get(k)` / `obj[k]` |
> | `df.head()` 대충 | `console.table(rows)` |
> | `requests.get(url).json()` | `(await fetch(url)).json()` |
>
> 타입 표기(`: string` 등)는 파이썬의 타입 힌트와 같은 역할이며 무시하고 읽어도 됩니다.

## 모델 v1 한 줄 요약

**경주 내 z-score 정규화 → 가중 합 → softmax 승리 확률.**

경마는 절대 능력이 아니라 *같은 경주 출전마끼리의 상대 우위* 게임이므로,
모든 피처를 경주 안에서만 표준화(z-score)해 비교합니다. 신마처럼 데이터가 없는 말은
z=0(경주 평균) — "모르면 중립" 원칙입니다.


## 0. 준비 — API 키 로드

저장소 루트 `.env.local`에서 KRA 서비스 키를 읽습니다.
(파이썬의 `dotenv.load_dotenv()`에 해당하는 코드를 직접 작성한 것)


In [ ]:
// .env.local 파싱 — 파이썬이라면: dict(l.split("=",1) for l in open(...) if "=" in l)
const envText = await Deno.readTextFile("../.env.local");
const env: Record<string, string> = {};
for (const line of envText.split("\n")) {
  const m = line.match(/^([A-Z_]+)=(.*)$/);
  if (m) env[m[1]] = m[2].replace(/^"|"$/g, ""); // 따옴표 제거
}

const KRA_KEY = env.KRA_SERVICE_KEY;
console.log("키 로드:", KRA_KEY ? `OK (${KRA_KEY.slice(0, 6)}…)` : "실패 — .env.local 확인!");

## 1. KRA Open API 미니 클라이언트

서비스의 `lib/kra/client.ts`가 처리하는 응답 정규화 규칙을 압축한 버전입니다.
(전체 규칙은 `docs/kra-openapi-reference.md` 참조)

핵심 함정 두 가지:
1. **결과가 1건이면 `item`이 객체(dict), 2건 이상이면 배열(list)** — 항상 배열로 정규화해야 함.
   파이썬이라면 `item if isinstance(item, list) else [item]`.
2. `resultCode "03"` = "데이터 없음"이지 오류가 아님 → 빈 배열 반환.


In [ ]:
const BASE = "https://apis.data.go.kr/B551015";

/** KRA API 호출 + envelope 정규화. authParam: 레거시 API만 "ServiceKey"(대문자) */
async function kra(api: string, params: Record<string, string | number>,
                   authParam: "serviceKey" | "ServiceKey" = "serviceKey"): Promise<any[]> {
  const search = new URLSearchParams({
    [authParam]: KRA_KEY, _type: "json", pageNo: "1", numOfRows: "300",
  });
  for (const [k, v] of Object.entries(params)) search.set(k, String(v));

  const res = await fetch(`${BASE}/${api}?${search}`);
  const body = (await res.json()).response;
  if (body.header.resultCode === "03") return []; // 데이터 없음 — 오류 아님
  if (body.header.resultCode !== "00") throw new Error(body.header.resultMsg);

  const item = body.body?.items?.item;
  if (item == null || body.body.items === "") return [];
  return Array.isArray(item) ? item : [item]; // dict → [dict] 정규화
}

// 동작 확인: 지난 개최일 서울 출마표 첫 행
const sample = await kra("API78/chulmainfo", { rccrs_cd: 1, race_dt: "20260816" });
console.log(`출마표 ${sample.length}행. 첫 행 마명: ${sample[0]?.hrnm}, raceNo: "${sample[0]?.raceNo}"`);
// 주의: raceNo가 숫자가 아니라 "제1경주" 문자열! (아래 매핑에서 숫자 추출)

## 2. 프로덕션 모델 import ⭐

**이 노트북의 핵심.** 서비스가 실제로 쓰는 채점 함수를 그대로 가져옵니다.
여기서 실험한 가중치가 좋으면 `model-v2.ts`로 승격시키면 끝 — 포팅 없음.


In [ ]:
import {
  scoreRace, WEIGHTS, SOFTMAX_SCALE, MODEL_VERSION,
  type HorseInput, type ScoredHorse,
} from "../lib/predict/model-v1.ts";

console.log("모델 버전:", MODEL_VERSION);
console.log("프로덕션 가중치:", WEIGHTS);
console.log("softmax 온도:", SOFTMAX_SCALE);

## 3. 원시 데이터 → 모델 입력(HorseInput) 매핑

서비스의 `lib/predict/features.ts`와 동일한 로직입니다.
KRA 응답의 지저분한 값들을 정리합니다:
- 레이팅 `"()"` = 신마(데이터 없음) → `null`
- 숫자가 문자열로 오는 경우 → `Number()` 변환
- 기수 성적은 트랙 전체를 **1회 호출**로 받아 이름으로 조인 (경주당 N회 호출 방지)


In [ ]:
/** 파이썬이라면: float(s) 실패 시 None 반환하는 헬퍼 */
const toNum = (v: unknown): number | null => {
  const n = Number(String(v ?? "").trim());
  return Number.isFinite(n) ? n : null;
};

const parseRating = (raw: unknown): number | null => {
  const s = String(raw ?? "").trim();
  if (!s || s === "()") return null; // 신마
  const m = s.match(/-?\d+(\.\d+)?/);
  return m ? Number(m[0]) : null;
};

/** "제1경주" → 1 */
const parseRaceNo = (raw: unknown): number => Number(String(raw).match(/\d+/)?.[0] ?? NaN);

/** 한 날짜의 (경주번호 → HorseInput[]) 맵 구성 */
async function loadRaces(meet: number, ymd: string): Promise<Map<number, HorseInput[]>> {
  const [card, jockeyRows] = await Promise.all([
    kra("API78/chulmainfo", { rccrs_cd: meet, race_dt: ymd }),
    kra("API11_1/jockeyResult_1", { meet }, "ServiceKey"), // 레거시: 대문자 S!
  ]);
  const jockeys = new Map(jockeyRows.map((j: any) => [j.jkName, j]));

  const races = new Map<number, HorseInput[]>();
  for (const row of card) {
    const raceNo = parseRaceNo(row.raceNo);
    if (!raceNo) continue;
    const jk = jockeys.get(String(row.jckyNm ?? "").trim());
    const horse: HorseInput = {
      gate: Number(row.gtno) || 0,
      name: row.hrnm,
      rating: parseRating(row.rating),
      burdWgt: toNum(row.burdWgt),
      wgtIndec: toNum(row.wgtIndec),
      jkWinRateY: toNum(jk?.winRateY),
      jkQnlRateY: toNum(jk?.qnlRateY),
    };
    if (!races.has(raceNo)) races.set(raceNo, []);
    races.get(raceNo)!.push(horse);
  }
  return races;
}

const races = await loadRaces(1, "20260816"); // 서울, 지난 일요일
console.log(`경주 수: ${races.size}, 제1경주 출전: ${races.get(1)?.length}두`);

## 4. 예측 실행해보기

한 경주를 채점하고 결과를 표로 확인합니다. `contributions`에 피처별 기여도가 들어 있어
"왜 이 말이 1순위인가"를 설명할 수 있습니다 (서비스 UI의 '근거 자세히 보기'가 이 값).


In [ ]:
const scored = scoreRace(races.get(1)!);

console.table(scored.map((h) => ({
  순위: h.rank,
  마명: h.name,
  "승리확률%": (h.prob * 100).toFixed(1),
  레이팅: h.rating ?? "결측",
  "기수승률%": h.jkWinRateY ?? "결측",
  "최대기여피처": [...h.contributions].sort((a, b) => Math.abs(b.value) - Math.abs(a.value))[0].labelKo,
})));

## 5. 백테스트 — 과거 개최일에서 적중률 측정

모델 개선의 **유일한 기준**입니다. 감이 아니라 이 숫자로 판단하세요.

- `top1`: 모델 1순위가 실제 1착 (무작위 기대치 ≈ 1/출전두수 ≈ 8~10%)
- `top3`: 실제 1착이 모델 1~3순위 안 (무작위 기대치 ≈ 25~30%)

⚠️ 날짜 수를 한 번에 크게 늘리면 KRA 레이트리밋에 걸릴 수 있습니다. 5~10일씩.


In [ ]:
// 날짜 유틸도 프로덕션 코드 재사용 (lib/kst.ts는 의존성 제로)
import { addDays, todayKst, weekdayOf } from "../lib/kst.ts";

/** meet 트랙의 최근 과거 개최일 n개 (요일 기반) */
function pastRaceDays(raceDays: number[], n: number): string[] {
  const days: string[] = [];
  let d = addDays(todayKst(), -1);
  while (days.length < n) {
    if (raceDays.includes(weekdayOf(d))) days.push(d);
    d = addDays(d, -1);
  }
  return days;
}

interface BacktestRow { date: string; races: number; top1: number; top3: number; }

async function backtest(meet: number, raceDays: number[], nDays: number,
                        opts: Parameters<typeof scoreRace>[1] = {}): Promise<BacktestRow[]> {
  const rows: BacktestRow[] = [];
  for (const ymd of pastRaceDays(raceDays, nDays)) {
    const [raceMap, results] = await Promise.all([
      loadRaces(meet, ymd),
      kra("API299/Race_Result_total", { meet, rc_date: ymd }),
    ]);
    // 경주별 실제 1착 마명
    const winners = new Map<number, string>();
    for (const r of results) if (Number(r.ord) === 1) winners.set(Number(r.rcNo), r.hrName);

    let top1 = 0, top3 = 0, judged = 0;
    for (const [raceNo, horses] of raceMap) {
      const winner = winners.get(raceNo);
      if (!winner) continue;
      judged++;
      const ranked = scoreRace(horses, opts);
      if (ranked[0]?.name === winner) top1++;
      if (ranked.slice(0, 3).some((h) => h.name === winner)) top3++;
    }
    rows.push({ date: ymd, races: judged, top1, top3 });
  }
  return rows;
}

function summarize(label: string, rows: BacktestRow[]) {
  const races = rows.reduce((a, r) => a + r.races, 0);
  const top1 = rows.reduce((a, r) => a + r.top1, 0);
  const top3 = rows.reduce((a, r) => a + r.top3, 0);
  console.log(`${label}: ${races}경주 — top1 ${(top1 / races * 100).toFixed(1)}%, top3 ${(top3 / races * 100).toFixed(1)}%`);
}

// 서울(토·일) 최근 4개 개최일
const base = await backtest(1, [6, 0], 4);
console.table(base);
summarize("기본 가중치(v1)", base);

## 6. 가중치 튜닝 실험

`scoreRace`의 두 번째 인자로 가중치/온도를 덮어써서 **프로덕션 코드 수정 없이** 실험합니다.
아래는 "레이팅을 더 믿어보면?" 실험 예시입니다. 다양한 조합을 돌려보고
5장의 백테스트와 **같은 기간**으로 비교하세요.


In [ ]:
// 실험: 레이팅 비중 강화 + 기수 비중 축소
const exp1 = await backtest(1, [6, 0], 4, {
  weights: { rating: 0.55, jkWinRateY: 0.1 },
});
summarize("실험1 rating↑", exp1);

// 실험: softmax 온도 — 확률 분포의 뾰족함 (top1 적중엔 영향 없고 확률 표시에만 영향)
// const exp2 = await backtest(1, [6, 0], 4, { scale: 4 });
// summarize("실험2 scale=4", exp2);

## 7. 개선 로드맵 (v2 후보 아이디어)

| 아이디어 | 데이터 소스 | 기대 효과 | 난이도 |
|---|---|---|---|
| **말 자체 성적** (승률/연대율 1년) | API15_2 (마명당 1회 — 캐시 필수) | ★★★ 현재 모델의 최대 공백 | 중 |
| 조교사 성적 | API308 | ★★ | 하 |
| 거리 적성 (해당 거리 과거 기록) | API214_1 월 단위 수집 | ★★★ | 상 |
| 최근 N회 착순 추세 (recentOrd 등) | API15_2 recent* 필드 | ★★ | 하 |
| 가중치를 데이터로 학습 (로지스틱 회귀) | 백테스트 데이터 축적 후 | ★★★ 수작업 튜닝 탈피 | 중 |
| 인기(배당) 대비 가치 베팅 지표 | API301 + 예측 확률 비교 | ★★ (다른 방향의 기능) | 중 |

### 개선을 서비스에 반영하는 절차

1. 이 노트북에서 실험 → 백테스트 개선 확인 (여러 기간에서!)
2. `lib/predict/model-v2.ts`를 **새 파일**로 작성 (v1 보존 — 버전 비교용)
3. `components/PredictView.tsx`의 import만 v2로 교체
4. 배포 후 서비스의 "이날 적중 성적" 섹션으로 실전 성능 계속 검증

### 주의사항

- **과최적화 경계**: 4일 백테스트에서 좋아 보여도 다른 기간에서 확인할 것
- **레이트리밋**: 백테스트는 캐시 없는 생 호출 — 한 세션에 수십 일씩 돌리지 말 것
- 신마 경주(레이팅 전원 결측)는 모델이 사실상 기수 성적만으로 판단함 — v2의 말 성적 피처가 해결
